# Matrix conversion: three small experiments
Place beside `OPFData_demo.ipynb`; use its Python environment and the revised `matrix_form.py` (with `NetworkEditor`). Run all cells. Six AC-OPF solves compare three originals with three edits.

Inputs are promoted to float64 before editing. Old solution labels are discarded; temporary flat labels only initialize the existing converter. Power is per unit, angles radians. Zero branch ratings mean **unmonitored**, matching PYPOWER.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from pypower.api import runopf, ppoption

root = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'distillation/conversion/convert.py').is_file())
sys.path.insert(0, str(root))
from distillation.acopf.opf_data import load_dataset
from distillation.conversion.convert import hetero_to_ppc
from distillation.conversion.matrix_form import (
    opf2mat, mat2opf, NetworkEditor, OPFSolution,
    validate_network, residuals, E_LINE, E_XFMR,
)

INDICES = [1, 50, 250]
TOL = 1e-5  # p.u. for power/voltage; radians for angles
options = ppoption(VERBOSE=0, OUT_ALL=0, OPF_VIOLATION=1e-7,
                   OPF_FLOW_LIM=0, OPF_IGNORE_ANG_LIM=False)
ds = load_dataset()
originals = {i: ds[i].clone().cpu().apply(
    lambda x: x.double() if x.is_floating_point() else x) for i in INDICES}


/opt/aurora/26.181.0/frameworks/aurora_frameworks-2026.1.0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def edit(data, experiment):
    mats = opf2mat(data)
    p = mats.params
    if experiment == 0:
        j = int(p.pd.argmax())
        p.pd[j] *= 1.05
        p.qd[j] *= 1.05
        name = 'largest load +5%'
    elif experiment == 1:
        j = int(p.pmax.argmax())
        p.pmax[j] -= 0.02 * (p.pmax[j] - p.pmin[j])
        name = 'largest generator: capacity range -2%'
    else:
        editor = NetworkEditor(mats)
        j = int(p.pmax.argmax())
        b = editor.add_bus(
            base_kv=float(p.base_kv[0]), vmin=float(p.vmin[0]), vmax=float(p.vmax[0]),
            load=dict(pd=0.02, qd=0.005),
            generator=dict(pmin=0, pmax=0.03, qmin=-0.02, qmax=0.02,
                           c2=float(p.c2[j]), c1=float(p.c1[j]), c0=0))
        editor.add_branch(0, b, br_r=0.01, br_x=0.1, rate_a=0.5,
                          angmin=-0.5, angmax=0.5)
        mats = editor.build()
        name = 'add connected bus + load + generator'
    validate_network(mats.params)
    return name, mat2opf(mats, include_solution=False)

experiments = {i: edit(originals[i], k) for k, i in enumerate(INDICES)}
modified_data = {i: data for i, (_, data) in experiments.items()}


Structural validation checks connectivity, reference buses and parameter bounds; it does **not** establish AC feasibility. A passive-network capacity shortage can rule out feasibility before solving. Otherwise, the solver searches for a feasible point.

The post-check evaluates AC balance, branch-flow consistency, thermal/angle limits, reference angle and voltage/generator bounds against the **edited matrix model**. Passing provides a numerical feasibility witness at `TOL`, not a formal certificate or proof of global optimality. Failure leaves feasibility unresolved. Cost changes compare local solver outcomes.


In [3]:
def solve_checked(data):
    mats = opf2mat(data)
    p = mats.params
    validate_network(p)
    if bool((p.br_r >= 0).all() and (p.gs >= 0).all()) and p.pmax.sum() < p.pd.sum() - TOL:
        return dict(status='infeasible: passive capacity shortage', feasible=False)

    seed = mat2opf(mats, include_solution=False)
    seed['bus'].y = torch.stack((torch.zeros_like(p.vmin),
                                torch.ones_like(p.vmin).clamp(p.vmin, p.vmax)), 1)
    seed['generator'].y = torch.stack((p.pg_set.clamp(p.pmin, p.pmax),
                                      p.qg_set.clamp(p.qmin, p.qmax)), 1)
    for et in (E_LINE, E_XFMR):
        seed[et].edge_label = torch.zeros((seed[et].edge_index.shape[1], 4), dtype=torch.float64)
    seed.objective = torch.tensor(0., dtype=torch.float64)
    ppc = hetero_to_ppc(seed)
    ids = ppc['bus'][:, 0]
    assert len(ids) == p.N and len(np.unique(ids)) == p.N
    assert np.array_equal(ppc['gen'][:, 0], ids[p.gen_bus.numpy()]), 'generator order changed'
    assert np.array_equal(ppc['branch'][:, :2], ids[torch.stack((p.fr, p.to), 1).numpy()]), 'branch order changed'
    result = runopf(ppc, options)
    assert np.array_equal(result['bus'][:, 0], ids), 'solver bus order changed'
    assert np.array_equal(result['gen'][:, 0], ppc['gen'][:, 0]), 'solver generator order changed'
    assert np.array_equal(result['branch'][:, :2], ppc['branch'][:, :2]), 'solver branch order changed'
    base = float(result['baseMVA'])
    t = lambda x: torch.as_tensor(np.asarray(x).copy(), dtype=torch.float64)
    mats.sol = OPFSolution(
        va=t(np.deg2rad(result['bus'][:, 8])), vm=t(result['bus'][:, 7]),
        pg=t(result['gen'][:, 1] / base), qg=t(result['gen'][:, 2] / base),
        pf=t(result['branch'][:, 13] / base), qf=t(result['branch'][:, 14] / base),
        pt=t(result['branch'][:, 15] / base), qt=t(result['branch'][:, 16] / base),
        objective=t(result['f']))
    checks = residuals(mats, zero_rating='unmonitored')
    keys = ['balance', 'flow_f', 'flow_t', 'thermal', 'angle', 'ref_angle',
            'v_bounds', 'pg_bounds', 'qg_bounds']
    feasible = all(np.isfinite(checks[k]) and checks[k] <= TOL for k in keys)
    cost_ok = checks['cost_abs'] <= 1e-6 * max(1, abs(result['f']))
    return dict(status='feasible witness' if feasible else 'unresolved: residual check failed',
                feasible=feasible, success=bool(result['success']), cost_ok=cost_ok,
                objective=float(result['f']), min_vm=float(mats.sol.vm.min()),
                max_residual=max(checks[k] for k in keys), checks=checks, raw=result,
                solved_data=mat2opf(mats) if feasible else None)


In [4]:
results, rows = {}, []
for i in INDICES:
    for variant, data in [('original', originals[i]), ('modified', modified_data[i])]:
        inputs = mat2opf(opf2mat(data), include_solution=False)
        try:
            r = solve_checked(inputs)
        except Exception as exc:
            r = dict(status=f'unresolved: {type(exc).__name__}: {exc}', feasible=False)
        results[i, variant] = r
        rows.append(dict(sample=i, variant=variant, edit=experiments[i][0],
                         **{k: r.get(k, np.nan) for k in
                            ['status', 'success', 'cost_ok', 'objective', 'min_vm', 'max_residual']}))
summary = pd.DataFrame(rows).set_index(['sample', 'variant'])
summary['cost_change_%'] = np.nan
for i in INDICES:
    a, b = results[i, 'original'], results[i, 'modified']
    if all(r.get('feasible') and r.get('success') and r.get('cost_ok') for r in (a, b)) and a['objective'] != 0:
        summary.loc[(i, 'modified'), 'cost_change_%'] = 100 * (b['objective'] - a['objective']) / abs(a['objective'])
display(summary)


edit            status  \
sample variant                                                             
1      original                       largest load +5%  feasible witness   
       modified                       largest load +5%  feasible witness   
50     original  largest generator: capacity range -2%  feasible witness   
       modified  largest generator: capacity range -2%  feasible witness   
250    original   add connected bus + load + generator  feasible witness   
       modified   add connected bus + load + generator  feasible witness   

                 success  cost_ok      objective    min_vm  max_residual  \
sample variant                                                             
1      original     True     True  100308.534571  0.969219  1.971411e-08   
       modified     True     True  100781.813368  0.969306  1.870630e-08   
50     original     True     True   95109.417593  0.991388  2.855484e-07   
       modified     True     True   95109.417607  0.991388  1.430970e-07   
250    original     True     True   98586.512794  0.981349  6.043132e-07   
       modified     True     True   98630.483882  0.981412  3.729878e-07   

                 cost_change_%  
sample variant                  
1      original            NaN  
       modified   4.718231e-01  
50     original            NaN  
       modified   1.418469e-08  
250    original            NaN  
       modified   4.460152e-02

`modified_data[i]` contains edited, unlabeled HeteroData inputs. `results[i, 'modified']['solved_data']` contains newly computed labels only when the feasibility check passes; `checks` and `raw` retain diagnostics and the solver output. Inspect `success` and `cost_ok` before interpreting objective changes. A feasible point from a nonconverged run is not an optimal solution.

Increase/decrease the constants in `edit()` and rerun from that cell. Each edit starts from its untouched original. Node removal/merging can be substituted using `NetworkEditor`; keep the network connected with one reference bus. This notebook assumes the converter preserves bus, generator and line-then-transformer row order, as in the demo. It does not claim formal infeasibility certification beyond the stated passive-capacity test.
